In [ ]:
from __future__ import annotations

import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

TOKEN_RE = re.compile(r"(?u)\b\w\w+\b")


In [ ]:
class ScratchTfidfVectorizer:
    def __init__(self, min_df=2, max_features=100_000, ngram_range=(1, 2)):
        self.min_df = min_df
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.vocabulary_: dict[str, int] = {}
        self.idf_: np.ndarray | None = None

    def _terms(self, document):
        tokens = TOKEN_RE.findall(str(document).lower())
        terms = []
        if self.ngram_range[0] <= 1 <= self.ngram_range[1]:
            terms.extend(tokens)
        if self.ngram_range[0] <= 2 <= self.ngram_range[1]:
            terms.extend(a + " " + b for a, b in zip(tokens, tokens[1:]))
        return terms

    def fit(self, documents):
        document_frequency = Counter()
        n_documents = 0

        for document in documents:
            document_frequency.update(set(self._terms(document)))
            n_documents += 1

        eligible = (
            (term, df)
            for term, df in document_frequency.items()
            if df >= self.min_df
        )

        selected = sorted(
            eligible,
            key=lambda item: (-item[1], item[0])
        )[: self.max_features]

        self.vocabulary_ = {
            term: index for index, (term, _) in enumerate(selected)
        }

        dfs = np.asarray([df for _, df in selected], dtype=np.float64)
        self.idf_ = np.log((1.0 + n_documents) / (1.0 + dfs)) + 1.0
        return self

    def transform(self, documents):
        if self.idf_ is None:
            raise RuntimeError("Vectorizer must be fitted before transform().")

        rows, columns, values = [], [], []

        for row, document in enumerate(documents):
            counts = Counter(self._terms(document))

            for term, count in counts.items():
                column = self.vocabulary_.get(term)

                if column is not None:
                    rows.append(row)
                    columns.append(column)
                    values.append(1.0 + np.log(count))

        matrix = sparse.csr_matrix(
            (
                np.asarray(values) * self.idf_[columns],
                (rows, columns),
            ),
            shape=(
                row + 1 if "row" in locals() else 0,
                len(self.vocabulary_),
            ),
            dtype=np.float64,
        )

        norms = np.sqrt(matrix.multiply(matrix).sum(axis=1)).A1
        norms[norms == 0.0] = 1.0

        return sparse.diags(1.0 / norms).dot(matrix).tocsr()

    def fit_transform(self, documents):
        documents = list(documents)
        return self.fit(documents).transform(documents)


In [ ]:
class ScratchLinearSVM:
    """Mini-batch gradient descent for a linear hinge-loss SVM."""

    def __init__(
        self,
        C=1.0,
        epochs=25,
        batch_size=256,
        learning_rate=0.5,
        decay=0.02,
        class_weight=None,
        random_state=42,
    ):
        self.C = C
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.decay = decay
        self.class_weight = class_weight
        self.random_state = random_state
        self.weights_: np.ndarray | None = None
        self.bias_ = 0.0

    def fit(self, X, y, verbose=False):
        y = np.asarray(y, dtype=np.int8)
        signed_y = np.where(y == 1, 1.0, -1.0)

        sample_weight = np.ones(len(y))

        if self.class_weight == "balanced":
            counts = np.bincount(y, minlength=2)
            sample_weight = np.asarray([
                len(y) / (2 * counts[label])
                for label in y
            ])

        self.weights_ = np.zeros(X.shape[1], dtype=np.float64)
        self.bias_ = 0.0

        regularization = 1.0 / (self.C * len(y))
        rng = np.random.default_rng(self.random_state)
        step = 0

        for epoch in range(self.epochs):
            order = rng.permutation(len(y))

            for start in range(0, len(y), self.batch_size):
                indices = order[start:start + self.batch_size]
                X_batch = X[indices]
                y_batch = signed_y[indices]
                weights = sample_weight[indices]

                margins = y_batch * (X_batch @ self.weights_ + self.bias_)
                violating = margins < 1.0

                eta = self.learning_rate / (1.0 + self.decay * step)

                # L2 regularisation gradient.
                gradient_w = regularization * self.weights_
                gradient_b = 0.0

                # Hinge-loss gradient for samples inside the margin.
                if np.any(violating):
                    coefficients = weights[violating] * y_batch[violating]

                    gradient_w -= np.asarray(
                        X_batch[violating].T @ coefficients
                    ).ravel() / len(indices)

                    gradient_b = -coefficients.sum() / len(indices)

                self.weights_ -= eta * gradient_w
                self.bias_ -= eta * gradient_b
                step += 1

            if verbose:
                print(
                    f"epoch={epoch + 1:02d} "
                    f"hinge_loss={self.hinge_loss(X, y):.6f}"
                )

        return self

    def decision_function(self, X):
        return np.asarray(
            X @ self.weights_ + self.bias_
        ).ravel()

    def predict(self, X):
        return (
            self.decision_function(X) >= 0.0
        ).astype(np.int8)

    def hinge_loss(self, X, y):
        signed_y = np.where(
            np.asarray(y) == 1,
            1.0,
            -1.0,
        )

        return np.maximum(
            0.0,
            1.0 - signed_y * self.decision_function(X),
        ).mean()


In [ ]:
def macro_f1(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    scores = []

    for label in (0, 1):
        tp = np.sum((y_true == label) & (y_pred == label))
        fp = np.sum((y_true != label) & (y_pred == label))
        fn = np.sum((y_true == label) & (y_pred != label))

        denominator = 2 * tp + fp + fn

        scores.append(
            0.0 if denominator == 0
            else 2 * tp / denominator
        )

    return float(np.mean(scores))


def load_split(train, split_path):
    split = pd.read_csv(split_path)

    # Support either the row_index-based split format or the id-based split
    # format used by some versions of the shared validation file.
    if "row_index" in split.columns:
        if (
            len(split) != len(train)
            or not np.array_equal(
                split["row_index"].to_numpy(),
                np.arange(len(train))
            )
        ):
            raise ValueError(
                "Shared split does not align with train.csv rows."
            )

    elif "id" in split.columns:
        if (
            len(split) != len(train)
            or train["id"].astype(str).tolist()
            != split["id"].astype(str).tolist()
        ):
            raise ValueError(
                "Shared split IDs do not align with train.csv."
            )

    else:
        raise ValueError(
            "shared_validation_split.csv must contain either "
            "'row_index' or 'id'."
        )

    train_mask = split["split"].eq("train").to_numpy()
    validation_mask = split["split"].eq("validation").to_numpy()

    return train_mask, validation_mask


In [ ]:
# Find the repository root automatically.
cwd = Path.cwd()

if (cwd / "data" / "train.csv").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "train.csv").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find data/train.csv. "
        "Open this notebook from the project repository or notebooks folder."
    )

TRAIN_PATH = PROJECT_ROOT / "data" / "train.csv"
TEST_PATH = PROJECT_ROOT / "data" / "test.csv"
SPLIT_PATH = PROJECT_ROOT / "data" / "splits" / "shared_validation_split.csv"
OUTPUT_PATH = PROJECT_ROOT / "submissions" / "SVM_FromScratch_Prediction.csv"

# Hyperparameters
C = 1.0
EPOCHS = 20
BATCH_SIZE = 256
LEARNING_RATE = 0.5
DECAY = 0.02
CLASS_WEIGHT = "balanced"
RANDOM_STATE = 42

print("Project root:", PROJECT_ROOT)
print("Output file:", OUTPUT_PATH)


In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train_mask, validation_mask = load_split(
    train,
    SPLIT_PATH,
)

print("Train rows:", int(train_mask.sum()))
print("Validation rows:", int(validation_mask.sum()))
print("Test rows:", len(test))


In [ ]:
vectorizer = ScratchTfidfVectorizer(
    min_df=2,
    max_features=100_000,
    ngram_range=(1, 2),
)

X_train = vectorizer.fit_transform(
    train.loc[train_mask, "text"].fillna("")
)

X_validation = vectorizer.transform(
    train.loc[validation_mask, "text"].fillna("")
)

model = ScratchLinearSVM(
    C=C,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    decay=DECAY,
    class_weight=CLASS_WEIGHT,
    random_state=RANDOM_STATE,
)

model.fit(
    X_train,
    train.loc[train_mask, "label"],
    verbose=True,
)

validation_predictions = model.predict(X_validation)

validation_score = macro_f1(
    train.loc[validation_mask, "label"],
    validation_predictions,
)

print(f"Validation Macro F1: {validation_score:.6f}")
print(
    "Validation prediction counts:",
    dict(zip(
        *np.unique(
            validation_predictions,
            return_counts=True,
        )
    )),
)


In [ ]:
final_vectorizer = ScratchTfidfVectorizer(
    min_df=2,
    max_features=100_000,
    ngram_range=(1, 2),
)

X_full = final_vectorizer.fit_transform(
    train["text"].fillna("")
)

X_test = final_vectorizer.transform(
    test["text"].fillna("")
)

final_model = ScratchLinearSVM(
    C=C,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    decay=DECAY,
    class_weight=CLASS_WEIGHT,
    random_state=RANDOM_STATE,
)

final_model.fit(
    X_full,
    train["label"],
    verbose=True,
)

test_predictions = final_model.predict(X_test)

print(
    "Test prediction counts:",
    dict(zip(
        *np.unique(
            test_predictions,
            return_counts=True,
        )
    )),
)


In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "label": test_predictions,
})

if len(submission) != len(test):
    raise RuntimeError("Submission row count does not match test.csv.")

if submission["label"].isna().any():
    raise RuntimeError("Submission contains missing predictions.")

if not submission["label"].isin([0, 1]).all():
    raise RuntimeError("Submission labels must be 0 or 1.")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
display(submission.head())
